# This script uses the Moveo Analytics API to fetch conversation data for specified Brain IDs

## Before you Begin
Make sure you have access to your Moveo Analytics API Key and Account ID, and that they are stored in a `.env` file.

## More Information
For more information about the Analytics API, please visit:
* https://docs.moveo.ai/docs/analytics/api_overview
* https://docs.moveo.ai/docs/analytics/log_sessions_content



In [2]:
import os
from dotenv import load_dotenv
import requests
from utils.common import execute_query, initialize_graphql_clinet
import pandas as pd

# Carregar variáveis do .env
load_dotenv()
GRAPHQL_API_KEY = os.getenv("MOVEO_API_KEY")
ACCOUNT_ID = os.getenv("MOVEO_ACCOUNT_ID")
AZURE_OPEN_AI_API_KEY = os.getenv("AZURE_OPEN_AI_API_KEY")

client = initialize_graphql_clinet(ACCOUNT_ID, GRAPHQL_API_KEY)

# Queries GraphQL
def get_sessions_query():
    return """
        query SessionsPreviewV2($brainIds: _uuid, $limit: Int) {
            rows: logs_sessions_preview_v2(limit: $limit, args: { brain_parent_ids: $brainIds, is_meaningful: true }) {
                session_id
            }
        }
    """

def get_session_content_query():
    return """
        query SessionContentV2($sessionId: String) {
            rows: log_session_content_v2(args: { session_id: $sessionId }) {
                messages
                session_id
            }
        }
    """

def fetch_sessions(brain_ids, limit=10):
    """Busca os session_ids para um determinado brain_id"""
    response = execute_query(client, get_sessions_query(), {"brainIds": brain_ids, "limit": limit})
    return [session["session_id"] for session in response]

def fetch_session_content(session_id):
    """Busca o conteúdo de uma sessão pelo session_id e formata as mensagens."""
    response = execute_query(client, get_session_content_query(), {"sessionId": session_id})
    
    if not response:
        return None
    
    messages = response[0]["messages"]
    formatted_messages = []
    for message in messages:
        if "event" in message and "message" in message:
            if message["event"] == "message:received" and "text" in message["message"]:
                formatted_messages.append(f"USER: {message['message']['text']}")
            elif message["event"] == "message:brain_send" and "responses" in message["message"]:
                bot_responses = []
                for response in message["message"]["responses"]:
                    if "text" in response:
                        bot_responses.append(response["text"])
                    else:
                        bot_responses.append(f"{response['type']} sent")
                formatted_messages.append("BOT: " + "\n".join(bot_responses))


    
    return {"session_id": session_id, "conversation": "\n".join(formatted_messages)}

def fetch_brain_conversations(brain_ids, limit=50):
    """Busca até 'limit' conversas para cada brain_id informado"""
    conversations = []
    session_ids = fetch_sessions(brain_ids, limit)
    print(session_ids)
    for session_id in session_ids:
        session_content = fetch_session_content(session_id)
        if session_content:
            conversations.append(session_content)
    return conversations



def call_gpt(session_id: str, prompt: str, max_tokens: int = 1024):
    url = "https://autoai-production-euswiss.openai.azure.com/openai/deployments/autoai-gpt4o/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPEN_AI_API_KEY
    }
    
    params = {
        "api-version": "2024-02-15-preview"
    }
    
    payload = {
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0.1,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0,
        "top_p": 0.95,
        "stop": None
    }
    
    print(f"Calling GPT to get department for session {session_id}")
    
    response = requests.post(url, json=payload, headers=headers, params=params)
    
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error {response.status_code}: {response.text}")
        response.raise_for_status()


def generate_conversation_summary(session_id: str, conversation: str) -> str:
    """
    Generate a summary of the conversation using GPT.
    
    Args:
        session_id (str): The session ID
        conversation (str): The full conversation text
        
    Returns:
        str: A summary of the conversation
    """
    prompt = f"""
    Please provide a concise summary of the following customer service conversation. 
    Focus on the main topic, customer's issue/request, and final resolution.
    
    Conversation:
    {conversation}
    """
    
    try:
        response = call_gpt(session_id, prompt)
        if response and 'choices' in response:
            return response['choices'][0]['message']['content'].strip()
        return "Failed to generate summary"
    except Exception as e:
        print(f"Error generating summary for session {session_id}: {str(e)}")
        return "Error generating summary"

# Modify fetch_brain_conversations to include summaries
def fetch_brain_conversations(brain_ids: str, limit: int = 50):
    conversations = []
    session_ids = fetch_sessions(brain_ids, limit)
    print(session_ids)
    for session_id in session_ids:
        session_content = fetch_session_content(session_id)
        if session_content:
            # Generate summary for the conversation
            summary = generate_conversation_summary(session_id, session_content)
            # Add both content and summary to the conversations list
            conversations.append({
                'session_id': session_id,
                'content': session_content,
                'summary': summary
            })
    return conversations

# Example usage remains the same
brain_ids = "{46a3690b-3ef3-46d1-969b-3819d7ae880c, 3dfa38a3-a097-40f8-a37c-adcece13743a}"  # Replace with real IDs
conversations = fetch_brain_conversations(brain_ids)
for conv in conversations:
    print(f"\nSession ID: {conv['session_id']}")
    print(f"Summary: {conv['summary']}")
    print("-" * 50)


    
# Create a DataFrame from the conversations
df = pd.DataFrame(conversations)

# Export to Excel
output_file = "conversation_summaries.xlsx"
df.to_excel(output_file, index=False)
print(f"\nResults exported to {output_file}")


['16c83a37-d5e2-46a8-a528-173fad3a82c5', '9de5128f-c515-4c97-8b1c-8b9e643b44b0', 'daf015d1-0aa9-4ce5-919f-8ed1efcfe67f', '1a6b282e-fb92-41b5-b789-bda497dafc18', '53d3c5dc-6245-4baf-b94e-aaf1be034d31', '2c90335a-52d2-4135-8e33-697fdb58a6f5', '7b35203f-e052-4ad2-9af7-317cbfab4bab', '14e15e5a-9bce-40f3-b6ad-f8c9f23165e2', '43a60a0d-7d2d-4300-be0b-0ae134b71aaa', '267558be-aeb5-4a6c-b112-6af917b078d7', '4262bd96-e3ee-4709-bbc5-0d80d27e0522', '59450946-6381-4dbe-9178-5f4cb1c91924', '8bf093e5-777d-49f2-b570-857d67e534c2', 'e04995b8-e5cd-40f1-acf0-6ec48231c796', 'c9c8bc27-3451-48aa-9e18-b964497334f6', '36af82c1-4864-4055-8292-3c70bd7de649', '150e523d-e56e-41c5-bf87-f102b8554c6e', 'c8edc8da-a2b2-4906-ab5f-d06aaf1b9116', '4d3f4c71-0c00-4519-8345-12025188f160', '737c0d55-bdd6-483c-9f66-1771b2ce8828', '6586943c-0d02-4448-a705-61f88736d6fe', 'b97226ff-585b-4830-b438-04bcce62994e', 'f4090f0b-6edd-4aa9-957d-89ea35d75ad9', '200aa22d-b255-40d8-a5fc-f0644a2b3c02', 'eae912ee-f871-4b2a-9c74-51db33a1a82c',